# Patient 1 – CSP + LDA Motor Imagery Classification

**Pipeline:** Notch + Bandpass → Epoch → CSP spatial filtering → LDA classification  
**Condition A:** Train on `P1_pre_training`, evaluate on `P1_pre_test`  
**Condition B:** Train on `P1_post_training`, evaluate on `P1_post_test`  
**Labels:** 0 = Right Hand, 1 = Left Hand

In [ ]:
import os
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch
import mne
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

mne.set_log_level('WARNING')

# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_DIR   = r"C:\Users\DIVI\Downloads\data"
FS         = 256          # sampling frequency (Hz)
N_CSP      = 4            # number of CSP components (2 per class)
TIME_AFTER = 2.0          # seconds after cue to start epoch
EPOCH_DUR  = 4.0          # epoch length in seconds
N_CV_FOLDS = 5            # stratified k-fold for cross-validation

CHANNEL_NAMES = ['FC3','FCz','FC4','C5','C3','C1','Cz','C2','C4','C6',
                 'CP3','CP1','CPz','CP2','CP4','Pz']

info = mne.create_info(ch_names=CHANNEL_NAMES, sfreq=FS, ch_types='eeg')
info.set_montage(mne.channels.make_standard_montage('standard_1020'))
print('Setup complete.')

In [ ]:
# ── PREPROCESSING FUNCTIONS ───────────────────────────────────────────────────

def notch_filter(data, fs, freq, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, data, axis=0)

def bandpass_filter(data, fs, lowcut=8.0, highcut=30.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data, axis=0)

def preprocess(y, fs=FS):
    """Remove powerline noise, then bandpass to motor imagery band (8–30 Hz)."""
    y = notch_filter(y, fs, 50)
    y = notch_filter(y, fs, 60)
    y = bandpass_filter(y, fs, 8, 30)
    return y

def get_trigger_onsets(trig):
    """Return sample indices where trigger transitions from 0 → nonzero."""
    t = trig.flatten()
    return np.where((t[1:] != 0) & (t[:-1] == 0))[0] + 1

def epoch_and_label(data, trig, fs=FS, time_after=TIME_AFTER, epoch_dur=EPOCH_DUR):
    """
    Cuts fixed-length epochs after each trigger and assigns class labels.

    Returns
    -------
    X : ndarray, shape (n_epochs, n_channels, n_times)   ← CSP expects this
    y : ndarray, shape (n_epochs,)   0 = right, 1 = left
    """
    trig_flat   = trig.flatten()
    onset_samps = get_trigger_onsets(trig_flat)
    start_off   = int(time_after * fs)
    n_samps     = int(epoch_dur * fs)

    epochs, labels = [], []
    for idx in onset_samps:
        # Determine class from sign of trigger value near onset
        window = trig_flat[max(0, idx - 2): idx + 3]
        label  = int(np.sign(np.sum(window)))

        start = idx + start_off
        end   = start + n_samps

        if end <= data.shape[0] and label != 0:
            epochs.append(data[start:end, :].T)          # → (channels, time)
            labels.append(0 if label == -1 else 1)       # 0=right, 1=left

    return np.array(epochs), np.array(labels)

In [ ]:
# ── LOAD ALL FOUR P1 FILES ────────────────────────────────────────────────────

def load_mat(fname):
    mat  = scipy.io.loadmat(os.path.join(DATA_DIR, fname))
    y    = mat['y'].astype(float)   # (samples, channels)
    trig = mat['trig']
    return y, trig

print('Loading raw data …')
y_pre_tr,  trig_pre_tr  = load_mat('P1_pre_training.mat')
y_pre_te,  trig_pre_te  = load_mat('P1_pre_test.mat')
y_post_tr, trig_post_tr = load_mat('P1_post_training.mat')
y_post_te, trig_post_te = load_mat('P1_post_test.mat')

print('Filtering (notch 50/60 Hz + bandpass 8–30 Hz) …')
y_pre_tr  = preprocess(y_pre_tr)
y_pre_te  = preprocess(y_pre_te)
y_post_tr = preprocess(y_post_tr)
y_post_te = preprocess(y_post_te)

print('Epoching …')
X_pre_tr,  L_pre_tr  = epoch_and_label(y_pre_tr,  trig_pre_tr)
X_pre_te,  L_pre_te  = epoch_and_label(y_pre_te,  trig_pre_te)
X_post_tr, L_post_tr = epoch_and_label(y_post_tr, trig_post_tr)
X_post_te, L_post_te = epoch_and_label(y_post_te, trig_post_te)

print()
print(f"{'Dataset':<18} {'Shape':>20}   right   left")
print('-' * 55)
for name, X, L in [
    ('pre_training',  X_pre_tr,  L_pre_tr),
    ('pre_test',      X_pre_te,  L_pre_te),
    ('post_training', X_post_tr, L_post_tr),
    ('post_test',     X_post_te, L_post_te),
]:
    print(f"{name:<18} {str(X.shape):>20}   {np.sum(L==0):>5}   {np.sum(L==1):>4}")

In [ ]:
# ── CSP + LDA: FIT & EVALUATE ─────────────────────────────────────────────────

def run_csp_lda(X_train, L_train, X_test, L_test, label=''):
    """
    Fits CSP+LDA on training data, evaluates on test data.
    Returns the fitted pipeline plus metrics dict.
    """
    pipe = Pipeline([
        ('csp', CSP(n_components=N_CSP, reg=None, log=True, norm_trace=False)),
        ('lda', LDA()),
    ])

    cv        = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
    cv_scores = cross_val_score(pipe, X_train, L_train, cv=cv, scoring='accuracy')

    pipe.fit(X_train, L_train)
    y_pred   = pipe.predict(X_test)
    acc_test = accuracy_score(L_test, y_pred)

    print(f'=== {label} ===')
    print(f'  5-fold CV on training : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
    print(f'  Test accuracy         : {acc_test:.3f}  ({acc_test*100:.1f}%)')
    print(f'  Chance level          : {max(np.mean(L_test==0), np.mean(L_test==1)):.3f}')
    print()

    return pipe, {'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
                  'test_acc': acc_test, 'y_pred': y_pred}

pipe_pre,  metrics_pre  = run_csp_lda(X_pre_tr,  L_pre_tr,  X_pre_te,  L_pre_te,  'PRE  CONDITION')
pipe_post, metrics_post = run_csp_lda(X_post_tr, L_post_tr, X_post_te, L_post_te, 'POST CONDITION')

In [ ]:
# ── CONFUSION MATRICES ────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, pipe, X_te, L_te, label, acc in [
    (axes[0], pipe_pre,  X_pre_te,  L_pre_te,  'Pre',  metrics_pre['test_acc']),
    (axes[1], pipe_post, X_post_te, L_post_te, 'Post', metrics_post['test_acc']),
]:
    cm = confusion_matrix(L_te, pipe.predict(X_te))
    disp = ConfusionMatrixDisplay(cm, display_labels=['Right', 'Left'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Patient 1 – {label} Training\nTest Accuracy: {acc*100:.1f}%', fontsize=12)

plt.suptitle('CSP + LDA Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('P1_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: P1_confusion_matrices.png')

In [ ]:
# ── CSP SPATIAL PATTERNS (topomaps) ──────────────────────────────────────────
#
# CSP patterns show WHICH scalp regions each component is sensitive to.
# Components 1–2 maximise variance for one class; last 2 for the other.

for label, pipe in [('Pre', pipe_pre), ('Post', pipe_post)]:
    csp      = pipe.named_steps['csp']
    patterns = csp.patterns_          # (n_components, n_channels)

    fig, axes = plt.subplots(1, N_CSP, figsize=(4 * N_CSP, 4))
    vmax = np.abs(patterns[:N_CSP]).max()

    for i, ax in enumerate(axes):
        mne.viz.plot_topomap(
            patterns[i], info,
            axes=ax, show=False, cmap='RdBu_r',
            vlim=(-vmax, vmax)
        )
        side = 'Right-dominant' if i < N_CSP // 2 else 'Left-dominant'
        ax.set_title(f'CSP {i+1}\n({side})', fontsize=9)

    plt.suptitle(f'Patient 1 – {label} Training: CSP Spatial Patterns', fontsize=13)
    plt.tight_layout()
    fname = f'P1_{label.lower()}_csp_patterns.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

In [ ]:
# ── PRE vs POST ACCURACY COMPARISON ──────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))

categories = [
    'Pre\nCV (train)', 'Pre\nTest',
    'Post\nCV (train)', 'Post\nTest'
]
values = [
    metrics_pre['cv_mean'],  metrics_pre['test_acc'],
    metrics_post['cv_mean'], metrics_post['test_acc'],
]
errors = [
    metrics_pre['cv_std'],  0,
    metrics_post['cv_std'], 0,
]
colors = ['#4C72B0', '#4C72B0', '#DD8452', '#DD8452']

bars = ax.bar(categories, values, yerr=errors, color=colors,
              alpha=0.82, capsize=6, width=0.5, edgecolor='white')

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Chance (50%)')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Patient 1 – CSP + LDA: Pre vs Post Rehabilitation', fontsize=13)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)

# Label each bar with percentage
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03,
        f'{val*100:.1f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

# Annotate improvement
delta = metrics_post['test_acc'] - metrics_pre['test_acc']
sign  = '+' if delta >= 0 else ''
ax.annotate(
    f'Test Δ = {sign}{delta*100:.1f}%',
    xy=(0.74, 0.88), xycoords='axes fraction',
    fontsize=11, color='#DD8452',
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#DD8452', alpha=0.8)
)

plt.tight_layout()
plt.savefig('P1_accuracy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: P1_accuracy_comparison.png')

In [ ]:
# ── SUMMARY ───────────────────────────────────────────────────────────────────

print('=' * 50)
print('  PATIENT 1 – CSP + LDA SUMMARY')
print('=' * 50)
print(f'  Epoch window  : {TIME_AFTER}s – {TIME_AFTER+EPOCH_DUR}s post-cue ({EPOCH_DUR}s)')
print(f'  Freq band     : 8–30 Hz (Mu + Beta)')
print(f'  CSP components: {N_CSP}')
print()
print(f"  {'Condition':<12} {'CV Acc':>10}  {'Test Acc':>10}")
print(f"  {'-'*36}")
print(f"  {'Pre'::<12} {metrics_pre['cv_mean']*100:>9.1f}%  {metrics_pre['test_acc']*100:>9.1f}%")
print(f"  {'Post'::<12} {metrics_post['cv_mean']*100:>9.1f}%  {metrics_post['test_acc']*100:>9.1f}%")
delta = metrics_post['test_acc'] - metrics_pre['test_acc']
sign  = '+' if delta >= 0 else ''
print(f"  {'Δ (post−pre)':<12} {'':>10}  {sign}{delta*100:>8.1f}%")
print('=' * 50)